In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from data import UncertainDataset, ds_single, ds_pressure

In [3]:
import fcbench

In [4]:
def compute_clean_bitinformation(xs: np.array) -> tuple[float, ...]:
    bit_information = fcbench.metrics.BitInformation.bit_information(
        xs, set_zero_insignificant_confidence=0.99,
    )
    assert len(bit_information) == 32

    # find the first per-bit information maximum in the mantissa
    # round to two decimal places to improve stability
    bit_information_mantissa = bit_information[9:]
    increasing = np.round(bit_information_mantissa[1:], 2) >= np.round(bit_information_mantissa[:-1], 2)
    imax = np.where(~increasing)[0]
    if len(imax) == 0:
        # no false information at the end
        return tuple(bit_information)
    imax = imax[0]

    # find the first per-bit information maximum in the mantissa
    # round to two decimal places to improve stability
    bit_information_skyfall = bit_information[9+imax:]
    decreasing = np.round(bit_information_skyfall[1:], 2) <= np.round(bit_information_skyfall[:-1], 2)
    imin = np.where(~decreasing)[0]
    if len(imin) == 0:
        # no false information at the end
        return tuple(bit_information)
    imin = imin[0]

    bit_information_corrected = np.copy(bit_information)
    bit_information_corrected[9+imax+imin+1:] = 0.0
    
    return tuple(bit_information_corrected)

In [5]:
from dataclasses import dataclass

@dataclass
class BitInformationBounds:
    keepbits: int
    absolute: float
    relative: float

In [6]:
def compute_bitinformation_bounds(da: xr.DataArray) -> BitInformationBounds:
    xs = da.values.flatten()

    bit_information = compute_clean_bitinformation(xs)

    if np.sum(bit_information) == 0.0:
        return BitInformationBounds(
            keepbits=0, absolute=np.inf, relative=np.inf,
        )

    keepbits = np.searchsorted(np.cumsum(bit_information) / np.sum(bit_information), 0.99) + 1
    keepbits_mantissa = keepbits - 9
    
    relative = np.exp2(-keepbits_mantissa)
    absolute = np.nanmin(np.abs(xs)[xs != 0.0]) * relative

    return BitInformationBounds(
        keepbits=keepbits, absolute=absolute, relative=relative,
    )

In [7]:
from tqdm import tqdm

In [ ]:
with open("bit-information-single.txt", "w") as f:
    for key in tqdm(ds_single.mean):
        print(f"{key}: ", end="", flush=True, file=f)
        print(f"{compute_bitinformation_bounds(ds_single.mean[key])!r}", flush=True, file=f)

  4%|▍         | 10/226 [01:03<23:58,  6.66s/it]

In [ ]:
with open("bit-information-pressure.txt", "w") as f:
    for key in tqdm(ds_pressure.mean):
        print(f"{key}:", flush=True, file=f)
        for pressure in ds_pressure.mean[key].pressure_level.values:
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(f"{compute_bitinformation_bounds(ds_pressure.mean[key].sel(pressure_level=pressure))!r}", flush=True, file=f)